In [1]:
"""Forecast uncertainty: empirical prediction intervals and error-band plots.

Input CSV columns: Actual, Predicted, Target, Horizon, Model
Run: python uncertainty_ana.py --input forecast_predictions.csv --level 0.90

Use out-of-sample or rolling-origin predictions for meaningful intervals.
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:

REQUIRED = ["Actual", "Predicted", "Target", "Horizon", "Model"]



In [3]:

def add_prediction_intervals(df, level=0.90):
    aliases = {
        "actual": "Actual",
        "predicted": "Predicted",
        "target": "Target",
        "horizon_minutes": "Horizon",
        "model": "Model",
    }
    df = df.rename(columns={c: aliases[c] for c in df.columns if c in aliases and aliases[c] not in df.columns})
    if not 0 < level < 1:
        raise ValueError("level must be between 0 and 1, for example 0.90")
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}. Required: {REQUIRED}")

    data = df.copy()
    for col in ["Actual", "Predicted"]:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=REQUIRED)
    if data.empty:
        raise ValueError("No valid forecast rows found.")

    data["Residual"] = data["Actual"] - data["Predicted"]
    data["Absolute_Error"] = data["Residual"].abs()
    keys = ["Target", "Horizon", "Model"]
    radius = (
        data.groupby(keys)["Absolute_Error"].quantile(level)
        .rename("Interval_Radius").reset_index()
    )
    data = data.merge(radius, on=keys, how="left")
    data["Lower"] = data["Predicted"] - data["Interval_Radius"]
    data["Upper"] = data["Predicted"] + data["Interval_Radius"]
    data["Covered"] = data["Actual"].between(data["Lower"], data["Upper"])
    data["Interval_Level"] = level
    data["Interval_Width"] = data["Upper"] - data["Lower"]
    return data




In [4]:


def summarize_confidence(data):
    keys = ["Target", "Horizon", "Model"]
    summary = data.groupby(keys).agg(
        Samples=("Actual", "size"),
        Empirical_Coverage=("Covered", "mean"),
        Mean_Interval_Width=("Interval_Width", "mean"),
    ).reset_index()
    summary["Empirical_Coverage (%)"] = summary["Empirical_Coverage"] * 100
    return summary.sort_values(["Target", "Horizon", "Mean_Interval_Width"])




In [5]:

def plot_error_bands(data, output_dir):
    plot_dir = Path(output_dir) / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    for (target, horizon, model), group in data.groupby(["Target", "Horizon", "Model"]):
        group = group.reset_index(drop=True)
        x = np.arange(len(group))
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.plot(x, group["Actual"], label="Actual")
        ax.plot(x, group["Predicted"], label="Forecast")
        ax.fill_between(
            x, group["Lower"].to_numpy(dtype=float),
            group["Upper"].to_numpy(dtype=float), alpha=0.2,
            label="Empirical prediction interval"
        )
        ax.set_title(f"{target} | Horizon: {horizon} | Model: {model}")
        ax.set_xlabel("Test sample (chronological order)")
        ax.set_ylabel(str(target))
        ax.grid(True, alpha=0.25)
        ax.legend()
        fig.tight_layout()
        safe = lambda value: "".join(
            ch if ch.isalnum() or ch in "-_." else "_" for ch in str(value)
        )
        fig.savefig(plot_dir / f"{safe(target)}_{safe(horizon)}_{safe(model)}.png", dpi=150)
        plt.close(fig)



In [6]:


def main():
    parser = argparse.ArgumentParser()
    default_candidates = [
        Path.cwd() / "model_outputs" / "test_predictions.csv",
        Path.cwd() / "dataProcess" / "model_outputs" / "test_predictions.csv",
    ]
    default_input = next(
        (candidate for candidate in default_candidates if candidate.is_file()),
        default_candidates[0],
    )
    parser.add_argument("--input", default=str(default_input), help="Forecast predictions CSV")
    parser.add_argument("--output-dir", default="uncertainty_results")
    parser.add_argument("--level", type=float, default=0.90)
    args, extra_args = parser.parse_known_args()
    if extra_args:
        is_kernel_args = (
            len(extra_args) == 1 and extra_args[0].startswith("--f=")
        ) or (
            len(extra_args) == 2 and extra_args[0] in {"-f", "--f"}
        )
        if not is_kernel_args:
            parser.error(f"unrecognized arguments: {' '.join(extra_args)}")

    source = Path(args.input)
    if not source.is_file():
        raise FileNotFoundError(f"CSV not found: {source}")

    data = add_prediction_intervals(pd.read_csv(source), level=args.level)
    summary = summarize_confidence(data)
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    data.to_csv(output / "predictions_with_intervals.csv", index=False)
    summary.to_csv(output / "uncertainty_summary.csv", index=False)
    plot_error_bands(data, output)
    print("\\nUncertainty and model confidence comparison:\\n", summary.to_string(index=False))
    print(f"\\nIntervals saved to: {(output / 'predictions_with_intervals.csv').resolve()}")
    print(f"Summary saved to: {(output / 'uncertainty_summary.csv').resolve()}")
    print(f"Plots saved in: {(output / 'plots').resolve()}")
    print("\\nCompare coverage and interval width together; narrower intervals alone do not imply better forecasts.")


if __name__ == "__main__":
    main()


\nUncertainty and model confidence comparison:\n           Target  Horizon                        Model  Samples  Empirical_Coverage  Mean_Interval_Width  Empirical_Coverage (%)
Redemption Count       15                 RandomForest     1293            0.899459           201.364674               89.945862
Redemption Count       15             GradientBoosting     1293            0.899459           208.057078               89.945862
Redemption Count       15                      XGBoost     1293            0.899459           209.622314               89.945862
Redemption Count       15             LinearRegression     1293            0.899459           214.336633               89.945862
Redemption Count       15 MovingAverage4TrainingValues     1293            0.987626           723.000000               98.762568
Redemption Count       15       NaiveLastTrainingValue     1293            0.989946           770.000000               98.994586
Redemption Count       30                      X